In [1]:
from embedder import Embedder

embed = Embedder()

print("Embedder loaded")


Embedder loaded


In [2]:
query_q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(query_q1)

print("Vector shape:", v.shape)
print("First value:", v[0])


Vector shape: (384,)
First value: -0.02058203437252893


In [3]:
options_q1 = [-0.31, -0.02, 0.12, 0.44]

closest_q1 = min(options_q1, key=lambda x: abs(x - v[0]))

print("Raw value:", v[0])
print("Closest answer:", closest_q1)

Raw value: -0.02058203437252893
Closest answer: -0.02


In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Documents:", len(documents))
print(documents[0].keys())
print(documents[0]["filename"])

Documents: 72
dict_keys(['content', 'filename'])
01-agentic-rag/lessons/01-intro.md


In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

target_doc = None

for doc in documents:
    if doc["filename"] == target_filename:
        target_doc = doc
        break

print(target_doc["filename"])
print(target_doc["content"][:500])

02-vector-search/lessons/07-sqlitesearch-vector.md
# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over ev


In [6]:
doc_vector = embed.encode(target_doc["content"])

similarity_q2 = v.dot(doc_vector)

print("Similarity:", similarity_q2)

Similarity: 0.36107027225589694


In [7]:
options_q2 = [0.07, 0.37, 0.68, 0.92]

closest_q2 = min(options_q2, key=lambda x: abs(x - similarity_q2))

print("Raw similarity:", similarity_q2)
print("Closest answer:", closest_q2)

Raw similarity: 0.36107027225589694
Closest answer: 0.37


In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print("Chunks:", len(chunks))
print(chunks[0].keys())
print(chunks[0]["filename"])
print(chunks[0]["start"])

Chunks: 295
dict_keys(['start', 'content', 'filename'])
01-agentic-rag/lessons/01-intro.md
0


In [9]:
from tqdm.auto import tqdm
import numpy as np

chunk_texts = [chunk["content"] for chunk in chunks]

batch_size = 50
chunk_vectors = []

for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    chunk_vectors.extend(batch_vectors)

X = np.array(chunk_vectors)

print("X shape:", X.shape)

  0%|          | 0/6 [00:00<?, ?it/s]

X shape: (295, 384)


In [10]:
scores = X.dot(v)

idx = np.argmax(scores)

print("Best score:", scores[idx])
print("Best filename:", chunks[idx]["filename"])
print("Best start:", chunks[idx]["start"])

Best score: 0.6489017718578813
Best filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Best start: 1000


In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

print("Vector index created")

Vector index created


In [12]:
query_q4 = "What metric do we use to evaluate a search engine?"

query_vector_q4 = embed.encode(query_q4)

vector_results_q4 = vindex.search(
    query_vector_q4,
    num_results=5
)

print("First result filename:", vector_results_q4[0]["filename"])
print(vector_results_q4[0])

First result filename: 04-evaluation/lessons/05-search-metrics.md
{'start': 0, 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly

In [13]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

print("Text index created")

Text index created


In [14]:
query_q5 = "How do I store vectors in PostgreSQL?"

In [15]:
query_vector_q5 = embed.encode(query_q5)

vector_results_q5 = vindex.search(
    query_vector_q5,
    num_results=5
)

vector_files_q5 = [doc["filename"] for doc in vector_results_q5]

print("VECTOR RESULTS:")
for doc in vector_results_q5:
    print(doc["filename"], "| start:", doc["start"])

VECTOR RESULTS:
02-vector-search/lessons/08-pgvector.md | start: 0
02-vector-search/lessons/08-pgvector.md | start: 3000
03-orchestration/lessons/05-rag.md | start: 2000
02-vector-search/lessons/08-pgvector.md | start: 1000
02-vector-search/lessons/08-pgvector.md | start: 2000


In [16]:
text_results_q5 = text_index.search(
    query_q5,
    num_results=5
)

text_files_q5 = [doc["filename"] for doc in text_results_q5]

print("TEXT RESULTS:")
for doc in text_results_q5:
    print(doc["filename"], "| start:", doc["start"])

TEXT RESULTS:
02-vector-search/lessons/02-embeddings.md | start: 4000
03-orchestration/lessons/05-rag.md | start: 1000
02-vector-search/lessons/01-intro.md | start: 0
03-orchestration/lessons/05-rag.md | start: 0
02-vector-search/lessons/01-intro.md | start: 1000


In [17]:
print("Vector files:", vector_files_q5)
print("Text files:", text_files_q5)

only_in_vector = [
    filename for filename in vector_files_q5
    if filename not in text_files_q5
]

print("Files in vector results but not in text results:")
print(only_in_vector)

Vector files: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Text files: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Files in vector results but not in text results:
['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [19]:
query_q6 = "How do I give the model access to tools?"

In [20]:
query_vector_q6 = embed.encode(query_q6)

vector_results_q6 = vindex.search(
    query_vector_q6,
    num_results=5
)

print("VECTOR RESULTS:")
for doc in vector_results_q6:
    print(doc["filename"], "| start:", doc["start"])

VECTOR RESULTS:
01-agentic-rag/lessons/01-intro.md | start: 2000
04-evaluation/lessons/02-ground-truth.md | start: 1000
01-agentic-rag/lessons/16-other-frameworks.md | start: 0
01-agentic-rag/lessons/15-frameworks.md | start: 2000
01-agentic-rag/lessons/13-function-calling.md | start: 4000


In [21]:
text_results_q6 = text_index.search(
    query_q6,
    num_results=5
)

print("TEXT RESULTS:")
for doc in text_results_q6:
    print(doc["filename"], "| start:", doc["start"])

TEXT RESULTS:
01-agentic-rag/lessons/14-agentic-loop.md | start: 0
01-agentic-rag/lessons/13-function-calling.md | start: 4000
01-agentic-rag/lessons/13-function-calling.md | start: 5000
01-agentic-rag/lessons/13-function-calling.md | start: 1000
04-evaluation/lessons/02-ground-truth.md | start: 3000


In [22]:
hybrid_results_q6 = rrf(
    [vector_results_q6, text_results_q6],
    k=60,
    num_results=5
)

print("HYBRID RESULTS:")
for doc in hybrid_results_q6:
    print(doc["filename"], "| start:", doc["start"])

HYBRID RESULTS:
01-agentic-rag/lessons/13-function-calling.md | start: 4000
01-agentic-rag/lessons/01-intro.md | start: 2000
01-agentic-rag/lessons/14-agentic-loop.md | start: 0
04-evaluation/lessons/02-ground-truth.md | start: 1000
01-agentic-rag/lessons/16-other-frameworks.md | start: 0


In [23]:
print("Q6 answer:", hybrid_results_q6[0]["filename"])

Q6 answer: 01-agentic-rag/lessons/13-function-calling.md


In [26]:
final_answers = {
    "Q1": "-0.02",
    "Q2": "0.37",
    "Q3": "02-vector-search/lessons/07-sqlitesearch-vector.md",
    "Q4": "04-evaluation/lessons/05-search-metrics.md",
    "Q5": "02-vector-search/lessons/08-pgvector.md",
    "Q6": "01-agentic-rag/lessons/13-function-calling.md",
}

final_answers


{'Q1': '-0.02',
 'Q2': '0.37',
 'Q3': '02-vector-search/lessons/07-sqlitesearch-vector.md',
 'Q4': '04-evaluation/lessons/05-search-metrics.md',
 'Q5': '02-vector-search/lessons/08-pgvector.md',
 'Q6': '01-agentic-rag/lessons/13-function-calling.md'}